# FloraBERT benchmark-integrity audit

Run this notebook with a Jupyter kernel connected to a Google Colab runtime from VS Code.

This is the first benchmark step before further model training. It downloads the public maize NAM expression dataset and maize promoter MLM files in the remote runtime, then:

- writes hash-only expression and MLM manifests;
- checks exact sequence overlap;
- checks reverse-complement overlap;
- optionally checks 0.8-identity clusters with MMseqs2;
- records an explicit inductive-benchmark verdict.

No Torch, Transformers, Accelerate, or model weights are installed. No raw data is committed to the repository.


## Configuration

By default:

- repository code is cloned to /content/florabert;
- the audit output/cache is written to /content/florabert_benchmark_audit;
- setting FLORABERT_DRIVE_ROOT places the audit output under that mounted Drive directory;
- the code branch is audit/benchmark-integrity;
- the exact hash audit runs automatically;
- MMseqs2 clustering is opt-in with FLORABERT_RUN_MMSEQS=1.

The cluster audit can be expensive for roughly 1.6 million combined records. If it is not run, the report explicitly remains partial.


In [ ]:
import os
from pathlib import Path
import subprocess
import sys

repo_url = os.environ.get(
    "FLORABERT_REPO_URL",
    "https://github.com/gurveersinghvirk/florabert.git",
)
repo_ref = os.environ.get(
    "FLORABERT_REPO_REF",
    "audit/benchmark-integrity",
)
repo_dir = Path(
    os.environ.get("FLORABERT_REPO_DIR", "/content/florabert")
).expanduser()

script_relative = Path(
    "scripts/0-data-loading-processing/audit_benchmark_integrity.py"
)
script_path = repo_dir / script_relative

if not script_path.is_file():
    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    print(f"Cloning {repo_url} branch {repo_ref} to {repo_dir}")
    subprocess.check_call(
        [
            "git",
            "clone",
            "--branch",
            repo_ref,
            "--single-branch",
            repo_url,
            str(repo_dir),
        ]
    )
else:
    print("Using existing repository checkout:", repo_dir)

if not script_path.is_file():
    raise FileNotFoundError(f"Audit script is missing: {script_path}")

drive_root = os.environ.get("FLORABERT_DRIVE_ROOT", "").strip()
audit_root = Path(
    os.environ.get(
        "FLORABERT_AUDIT_ROOT",
        str(Path(drive_root) / "florabert_benchmark_audit")
        if drive_root
        else "/content/florabert_benchmark_audit",
    )
).expanduser()

audit_root.mkdir(parents=True, exist_ok=True)

print("Repository:", repo_dir.resolve())
print("Repository ref:", repo_ref)
print("Audit root:", audit_root.resolve())
print("No model-training dependencies will be installed.")


In [ ]:
# Install only the lightweight remote data-reader packages if absent.
# Do not install torch or other model-training packages.

import importlib.util
import subprocess
import sys

requirements = {
    "huggingface_hub": "huggingface_hub>=0.23",
    "datasets": "datasets>=2.18",
}

missing = [
    requirement
    for module, requirement in requirements.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing remote audit dependencies:", missing)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing]
    )
else:
    print("Lightweight audit dependencies are already available.")

print("Torch installation skipped.")


In [ ]:
# Run the complete exact/reverse-complement audit.
# Set FLORABERT_RUN_MMSEQS=1 before this cell only after providing MMseqs2
# in the remote runtime. The script will then run the combined cluster audit.

environment = os.environ.copy()
environment["FLORABERT_AUDIT_ROOT"] = str(audit_root)

command = [
    sys.executable,
    str(script_path),
    "--audit-root",
    str(audit_root),
]

print("Running:", " ".join(command))
subprocess.run(
    command,
    cwd=str(repo_dir),
    env=environment,
    check=True,
)


In [ ]:
import json

summary_path = audit_root / "audit_summary.json"
if not summary_path.is_file():
    raise FileNotFoundError(f"Audit summary was not created: {summary_path}")

with summary_path.open("r", encoding="utf-8") as handle:
    audit_summary = json.load(handle)

print("Strict inductive benchmark ready:",
      audit_summary["strict_inductive_benchmark_ready"])
print()
print("Exact overlap:")
print(json.dumps(audit_summary["exact_overlap"], indent=2))
print()
print("Cluster audit:")
print(json.dumps(audit_summary["cluster_audit"], indent=2))
print()
print("Interpretation:")
print(audit_summary["interpretation"])
print()
print("Reports:")
for path in sorted(audit_root.iterdir()):
    print(" ", path.name)


## How to interpret the result

- Any exact or reverse-complement overlap with downstream evaluation/test data must be reported.
- An overlap in all_seqs_train.txt means direct unsupervised exposure during MLM training.
- An overlap only in all_seqs_test.txt is still relevant if that file influenced checkpoint selection.
- Exact non-overlap is not evidence of 0.8-identity cluster non-overlap.
- If the cluster audit is not complete, do not call the benchmark inductive.
- If overlap is found, construct a second MLM corpus with affected downstream sequences/clusters removed. Retain the original corpus only as an explicitly labelled transductive-DAPT condition.

The audit output directory is the artefact to archive with the Git commit, dataset revisions, environment information, and later model results.
